# AI Factory Custom Model Fine-Tuning & Serving Server

This notebook contains the complete pipeline to:
1. Mount Google Drive and install dependencies.
2. Fine-tune `Llama-3.2-3B-Instruct` using QLoRA (Unsloth) on financial transcripts.
3. Save the trained LoRA adapter weights back to Google Drive.
4. Expose the model through an **OpenAI-Compatible FastAPI server** tunneled with **ngrok** back to your local machine.

## Step 1: Mount Google Drive and Install Packages

In [ ]:
# Mount Google Drive to persist weights and dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install fast-serving and fine-tuning dependencies
!pip install -q fastapi uvicorn pyngrok pydantic nest-asyncio datasets
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## Step 2: Initialize Base Model (Unsloth Llama-3.2-3B)

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True # 4bit quantization to run easily on free T4 GPU

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # Unsloth's non-gated mirror of Llama-3.2-3B-Instruct
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# Set up LoRA adapter settings
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq = None,
)

## Step 3: Load and Format Training Dataset

First, make sure you generated `colab_training_dataset.jsonl` locally using `generate_training_data.py` and uploaded it to your Google Drive in a folder named `models/ai_factory/`.

In [ ]:
# Copy dataset from drive
!mkdir -p data
!cp /content/drive/MyDrive/models/ai_factory/colab_training_dataset.jsonl ./data/

from datasets import load_dataset
dataset = load_dataset("json", data_files="data/colab_training_dataset.jsonl", split="train")

def format_prompts(examples):
    formatted = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted.append(text)
    return {"text": formatted}

dataset = dataset.map(format_prompts, batched=True)

## Step 4: Fine-Tune using QLoRA

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 40, # Short training step suitable for demo datasets
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

## Step 5: Save Fine-Tuned Adapter to Google Drive

In [ ]:
model.save_pretrained_lora("/content/drive/MyDrive/models/ai_factory/llama_3_2_3b_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/models/ai_factory/llama_3_2_3b_lora")
print("Adapter weights successfully saved to Google Drive!")

## Step 6: Expose API Server via ngrok Tunnel

First, upload your copy of `colab_server.py` to the Colab workspace (or paste it into a file using the cell below).

In [ ]:
# Write colab_server.py if not already uploaded
server_code = """
# ... (colab_server.py code here)
"""
# Ensure colab_server.py is copied from drive or local folder if uploaded
!cp /content/drive/MyDrive/models/ai_factory/colab_server.py . || echo "colab_server.py not found in drive, please upload it to Colab workspace."

In [ ]:
import nest_asyncio
from pyngrok import ngrok

# Paste your free ngrok Authtoken here
# Signup at ngrok.com to get your token
NGROK_TOKEN = ""
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Start ngrok HTTP tunnel on port 8000
public_url = ngrok.connect(8000)
print("\n" + "="*60)
print(f"COLAB TUNNEL API PUBLIC URL: {public_url.public_url}")
print("="*60 + "\n")
print("Copy the URL above and paste it into your local .env as COLAB_TUNNEL_URL=...")

# Apply nest_asyncio to run uvicorn in the notebook
nest_asyncio.apply()

# Launch server exposing the fine-tuned model
from colab_server import run_server
run_server("/content/drive/MyDrive/models/ai_factory/llama_3_2_3b_lora", port=8000)